# Challenge Day (4/21)

End-to-end workflow for the in-class identification challenge. The TA hands out 5 unknown feature vectors (a mix of 512-d ArcFace and 1000-d VGG19 pre-softmax logits). We take 1–3 photos per classmate, drop them into Drive as `{N}Gallery/` folders, rebuild the gallery, then run each unknown through the matcher that fits its dimensionality.

**Runtime:** set **Runtime → Change runtime type → GPU (T4)**.

All source patches from challenge day are already merged into `main` — numeric folder stems (`1Gallery` … `38Gallery`) parse automatically, the gallery walker picks up both `Gallery/` and `Gallery /`, and the loaders handle TA's pickled dict-wrapped `.npy` format.

## 1. Clone repo and install

In [ ]:
GITHUB_URL = 'https://github.com/ekw4tu/DS3001-Project.git'

!git clone $GITHUB_URL /content/DS3001-Project 2>/dev/null || (cd /content/DS3001-Project && git pull)
%cd /content/DS3001-Project
!pip install -q -r requirements.txt
!pip uninstall -y -q onnxruntime && pip install -q onnxruntime-gpu

## 2. Mount Drive

`FR_DATA_ROOT` points at the shared picture folder so the pipeline reads straight from Drive. Classmates added during class live as `{N}Gallery/` siblings of the original celebrity folders.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.environ['FR_DATA_ROOT'] = '/content/drive/MyDrive/DS_NN_Project_Pictures'
os.environ['FR_ARTIFACTS_DIR'] = '/content/drive/MyDrive/facial_recognition_artifacts'
!ls "$FR_DATA_ROOT"

## 3. Build the gallery (celebrities + classmates)

Picks up every `{N}Gallery/` classmate folder alongside the 10 original celebrities. Expect ~180 vectors across ~48 identities once the room is photographed.

In [ ]:
!python scripts/build_gallery.py --gpu

## 4. Receive the unknowns

Upload the `.npy` files the TA hands out to `/content/` (Colab file panel → Upload). Then sniff each one's dimensionality so we know which matcher to use:

- **512-d** → ArcFace → `identify_unknowns.py`
- **1000-d** → VGG19 pre-softmax logits → `match_vgg_logits.py`

In [ ]:
import glob
import numpy as np

def sniff(path):
    arr = np.load(path, allow_pickle=True)
    if arr.dtype == object:
        obj = arr.item() if arr.shape == () else arr
        if isinstance(obj, dict):
            for k in ('embeddings', 'features', 'vectors'):
                if k in obj:
                    return np.asarray(obj[k]).reshape(-1).shape[0]
    return np.asarray(arr).reshape(-1).shape[0]

for p in sorted(glob.glob('/content/*.npy')):
    print(f'{p}: dim={sniff(p)}')

## 5. Match the ArcFace unknowns (512-d)

Cosine-match against the full gallery (celebrities + classmates). Edit the filenames below to match what the TA hands out.

In [ ]:
!python scripts/identify_unknowns.py /content/ID1.npy --top-k 5
!python scripts/identify_unknowns.py /content/ID2.npy --top-k 5
!python scripts/identify_unknowns.py /content/ID3.npy --top-k 5

## 6. Match the VGG19 unknowns (1000-d pre-softmax logits)

**Why a separate script:** the TA's VGG vectors are the 1000-d `predictions`-layer output (pre-softmax logits), not the 4096-d `fc2` features our pipeline uses by default. `match_vgg_logits.py` extracts matching 1000-d logits (`fc2 @ W + b`) from every gallery image and cosine-matches.

**`--classmates-only`:** drops the 10 original celebrity identities so the dense celebrity centroids don't drown out the sparse classmate ones. The TA's challenge vectors are always classmates.

In [ ]:
!python scripts/match_vgg_logits.py /content/ID4.npy /content/ID5.npy \
    --classmates-only --gpu --top-k 5